# MethaneScope AI — Lesson 2.9: Safe Scheduling and Retry Logic

**Your verified starting point:** 1 NASA EMIT granule, 1 PostGIS feature, 16 automated tests passing. This lesson adds safe *opt-in* scheduling controls. Nothing is scheduled automatically.

### Objectives
1. Understand batch-processing safety and idempotency. 2. Design retries for temporary database outages. 3. Prevent concurrent scheduled runs on the same local project directory. 4. Preserve reproducibility and scientific provenance.

## 1 — Safe setup

Launch JupyterLab from your MethaneScope project directory. Use the **Python (MethaneScope)** kernel if registered. Docker must be running for a real ingestion; dry-run planning only inspects files.

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
print('Project:', ROOT)
assert (ROOT / 'methanescope' / 'scheduler.py').is_file(), 'Extract the Lesson 2.9 ZIP into your project root first.'

## 2 — Inspect the accepted manifest

We do not invent any new methane measurements. The accepted manifest originates from Lesson 2.3 and is revalidated during ingestion.

In [ ]:
from methanescope.scheduler import plan
info = plan(ROOT)
print(json.dumps(info, indent=2))
assert info['accepted_rows'] >= 1

## 3 — Exponential backoff

When a database is briefly unavailable, repeated immediate requests can worsen congestion. Backoff gives it time to recover. **We only retry DB-only preflight failures, not invalid data or scientific conflicts.**

In [ ]:
import pandas as pd
from methanescope.scheduler import exponential_wait
waits = [exponential_wait(i, base_seconds=2) for i in range(1,6)]
display(pd.DataFrame({'retry': range(1,6), 'wait_seconds': waits}))

## 4 — Run in dry-run mode

**This does not insert any data.** In the command line: `python -m methanescope.scheduler run`.

In [ ]:
from methanescope.scheduler import run_safely
result = run_safely(ROOT, execute=False)
print(json.dumps(result, indent=2))
assert result['result'] == 'DRY_RUN'

## 5 — Local concurrency protection

The atomic `.ingestion.lock` prevents two **scheduler** runs from using the same local project simultaneously. It is NOT a lock for the old direct ingestion command and is NOT a multi-machine distributed lease. We test it without using the database.

In [ ]:
from tempfile import TemporaryDirectory
from methanescope.scheduler import exclusive_project_lock, LockBusyError
with TemporaryDirectory() as temp:
    with exclusive_project_lock(Path(temp)):
        try:
            with exclusive_project_lock(Path(temp)):
                pass
        except LockBusyError:
            print('PASS: overlapping scheduler run blocked safely')

## 6 — Automated tests (run from Git Bash)

```bash
python -m pytest -q tests/test_science.py tests/test_monitor.py tests/test_reliability.py tests/test_scheduler.py
```

Expect **25 passing tests** if your existing 16 tests remain intact (16 + 9 new). If they do not pass, stop before using `--execute`.

## 7 — Optional real run after tests pass

Do this manually in Git Bash, not automatically from this notebook:

```bash
python -m methanescope.scheduler run --execute
python -m methanescope.pipeline status
```

Expected with existing NASA observation: monitored `SUCCESS`, underlying `SKIPPED`, PostGIS still showing 1 granule + 1 feature. The code will create a local audit under `outputs/scheduler/`.

## 8 — Study questions

1. Why do we NOT retry a failed checksum validation?
2. Explain the difference between scientific uncertainty and engineering integrity checks.
3. What danger exists when two scheduled pipeline runs overlap?
4. When should we adopt a PostgreSQL advisory lock or cloud lease rather than a local lock file?
5. Why is a 2022 satellite observation not necessarily an active emitter in October 2026?

**Next, Lesson 2.10:** Acquire and validate additional genuine NASA EMIT granules, then compare observations across space and time. This is necessary before credible source-persistency estimation.